# Kaggle histopathology - stronger OOD baseline

This notebook replaces the original frozen DINOv2 + linear probe baseline with a more
robust pipeline for center shift:

1. **DINOv3 ViT-S/16** backbone (still small enough for a 16 GB GPU).
2. **On-the-fly color / stain augmentation** to reduce center-specific reliance.
3. **Center-balanced sampling** so the model does not overfit the largest source center.
4. **Domain-adversarial training** using the known training center metadata.
5. **Threshold search on validation** because the challenge metric is accuracy, not BCE.
6. **Test-time augmentation** during submission.


In [1]:
import os
import gc
import copy
import math
import time
import random
import warnings

import h5py
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import functional as TF
from transformers import AutoModel

warnings.filterwarnings("ignore")

In [10]:
TRAIN_IMAGES_PATH = "Data/train.h5"
VAL_IMAGES_PATH = "Data/val.h5"
TEST_IMAGES_PATH = "Data/test.h5"

SEED = 0
MODEL_NAME = "facebook/dinov3-vits16-pretrain-lvd1689m"

IMAGE_SIZE = 224
BATCH_SIZE = 32               # safe default for a 16 GB card with AMP
NUM_WORKERS = 0
ACCUMULATION_STEPS = 1

NUM_EPOCHS = 10
PATIENCE = 3
LR_BACKBONE = 1e-5
LR_HEAD = 2e-4
WEIGHT_DECAY = 1e-4

DOMAIN_ADV_WEIGHT = 0.20      # set to 0.0 to disable center adversarial loss
LABEL_SMOOTHING = 0.02
USE_TTA = True
TTA_MODES = ("none", "hflip", "vflip", "hvflip")

SAVE_PATH = "best_dinov3_ood.pth"
SUBMISSION_PATH = "submission_dinov3_ood.csv"

In [11]:
def seed_everything(seed: int = 0) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = device.type == "cuda"
print(f"Working on {device} | AMP={amp_enabled}")

Working on cuda | AMP=True


## Dataset helpers

In [12]:
def normalize_img(img: torch.Tensor) -> torch.Tensor:
    img = img.float()
    if img.max() > 1.5:
        img = img / 255.0
    return img.clamp(0.0, 1.0)

def imagenet_normalize(img: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=img.dtype, device=img.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], dtype=img.dtype, device=img.device).view(3, 1, 1)
    return (img - mean) / std

def random_stain_augment(img: torch.Tensor) -> torch.Tensor:
    """
    Lightweight stain / color perturbation on a [0, 1] tensor in CxHxW format.
    The goal is not to be histology-perfect, but to deliberately weaken center-specific color cues.
    """
    # Per-channel affine jitter
    scales = 1.0 + (torch.rand(3) * 0.36 - 0.18)
    shifts = (torch.rand(3) * 0.16 - 0.08)
    img = img * scales[:, None, None] + shifts[:, None, None]

    # Contrast / brightness
    brightness = random.uniform(0.85, 1.15)
    contrast = random.uniform(0.85, 1.20)
    mean = img.mean(dim=(1, 2), keepdim=True)
    img = (img - mean) * contrast + mean
    img = img * brightness

    # Mild gamma
    if random.random() < 0.7:
        gamma = random.uniform(0.85, 1.15)
        img = img.clamp(0.0, 1.0).pow(gamma)

    img = img.clamp(0.0, 1.0)

    # Occasional blur to reduce scanner-specific high-frequency cues
    if random.random() < 0.20:
        kernel = random.choice([3, 5])
        sigma = random.uniform(0.1, 1.2)
        img = TF.gaussian_blur(img, kernel_size=[kernel, kernel], sigma=[sigma, sigma])

    return img.clamp(0.0, 1.0)

class TrainTransform:
    def __init__(self, image_size: int = 224):
        self.image_size = image_size

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        img = normalize_img(img)

        # Resize first so geometry params are stable
        img = TF.resize(img, [self.image_size, self.image_size], antialias=True)

        # Symmetry-aware histology augmentations
        if random.random() < 0.5:
            img = torch.flip(img, dims=[2])
        if random.random() < 0.5:
            img = torch.flip(img, dims=[1])

        # Rotations preserve semantics for patches
        k = random.randint(0, 3)
        if k:
            img = torch.rot90(img, k=k, dims=(1, 2))

        # Small translations / scale jitter
        tx = int(random.uniform(-0.05, 0.05) * self.image_size)
        ty = int(random.uniform(-0.05, 0.05) * self.image_size)
        scale = random.uniform(0.95, 1.05)
        img = TF.affine(
            img,
            angle=0.0,
            translate=[tx, ty],
            scale=scale,
            shear=[0.0, 0.0],
            interpolation=TF.InterpolationMode.BILINEAR,
        )

        img = random_stain_augment(img)
        img = imagenet_normalize(img)
        return img

class EvalTransform:
    def __init__(self, image_size: int = 224):
        self.image_size = image_size

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        img = normalize_img(img)
        img = TF.resize(img, [self.image_size, self.image_size], antialias=True)
        img = imagenet_normalize(img)
        return img

In [13]:
class H5PatchDataset(Dataset):
    """
    HDF5 dataset with lazy opening per worker.
    """
    def __init__(self, dataset_path: str, transform=None, return_label: bool = True, return_center: bool = False):
        self.dataset_path = dataset_path
        self.transform = transform
        self.return_label = return_label
        self.return_center = return_center
        self._hdf = None

        self.ids = []
        self.labels = []
        self.centers = []

        with h5py.File(self.dataset_path, "r") as hdf:
            self.ids = sorted(list(hdf.keys()), key=lambda x: int(x))
            for img_id in self.ids:
                grp = hdf[img_id]
                if "label" in grp:
                    label = int(np.array(grp["label"]).reshape(-1)[0])
                else:
                    label = -1
                if "metadata" in grp:
                    center = int(np.array(grp["metadata"]).reshape(-1)[0])
                else:
                    center = -1

                self.labels.append(label)
                self.centers.append(center)

        self.labels = np.asarray(self.labels, dtype=np.int64)
        self.centers = np.asarray(self.centers, dtype=np.int64)

    def _get_hdf(self):
        if self._hdf is None:
            self._hdf = h5py.File(self.dataset_path, "r")
        return self._hdf

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_hdf"] = None
        return state

    def __del__(self):
        if getattr(self, "_hdf", None) is not None:
            try:
                self._hdf.close()
            except Exception:
                pass

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx: int):
        hdf = self._get_hdf()
        img_id = self.ids[idx]
        grp = hdf[img_id]

        img = torch.tensor(np.array(grp["img"]), dtype=torch.float32)
        if self.transform is not None:
            img = self.transform(img)

        out = {"image": img, "id": int(img_id)}

        if self.return_label:
            out["label"] = int(self.labels[idx])
        if self.return_center:
            out["center"] = int(self.centers[idx])

        return out

In [14]:
def compute_balanced_sample_weights(labels: np.ndarray, centers: np.ndarray) -> torch.DoubleTensor:
    """
    Balance on (center, label) so each source center contributes similarly and class balance
    is maintained inside each center.
    """
    unique_pairs, counts = np.unique(np.stack([centers, labels], axis=1), axis=0, return_counts=True)
    pair_to_count = {tuple(pair.tolist()): int(count) for pair, count in zip(unique_pairs, counts)}
    weights = np.array([1.0 / pair_to_count[(int(c), int(y))] for c, y in zip(centers, labels)], dtype=np.float64)
    weights = weights / weights.mean()
    return torch.as_tensor(weights, dtype=torch.double)

## Model

In [15]:
class GradReverseFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x: torch.Tensor, lambd: float = 1.0) -> torch.Tensor:
    return GradReverseFn.apply(x, lambd)

class OODDinov3Classifier(nn.Module):
    def __init__(self, model_name: str, n_centers: int):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name, output_hidden_states=True)

        try:
            self.backbone.gradient_checkpointing_enable()
        except Exception:
            pass

        hidden = self.backbone.config.hidden_size
        feat_dim = hidden * 2  # CLS + mean patch tokens

        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(512, 1),
        )

        self.domain_head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, n_centers),
        )

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(pixel_values=x, output_hidden_states=True)
        hs = out.hidden_states[-4:]

        cls = torch.stack([h[:, 0] for h in hs], dim=0).mean(dim=0)

        # Some ViT variants always expose patch tokens after the CLS token.
        # If ever unavailable, we fall back to CLS only.
        if hs[-1].shape[1] > 1:
            patch_mean = torch.stack([h[:, 1:].mean(dim=1) for h in hs], dim=0).mean(dim=0)
        else:
            patch_mean = cls

        feats = torch.cat([cls, patch_mean], dim=1)
        return feats

    def forward(self, x: torch.Tensor, grl_lambda: float = 0.0):
        feats = self.extract_features(x)
        cls_logits = self.classifier(feats).squeeze(1)
        dom_logits = self.domain_head(grad_reverse(feats, grl_lambda))
        return cls_logits, dom_logits

## Training / evaluation utilities

In [16]:
def bce_with_label_smoothing(logits: torch.Tensor, target: torch.Tensor, smoothing: float = 0.0) -> torch.Tensor:
    if smoothing > 0:
        target = target * (1.0 - smoothing) + 0.5 * smoothing
    return F.binary_cross_entropy_with_logits(logits, target)

@torch.no_grad()
def sweep_best_threshold(logits: np.ndarray, labels: np.ndarray):
    probs = 1.0 / (1.0 + np.exp(-logits))
    thresholds = np.linspace(0.05, 0.95, 181)
    best_thr = 0.5
    best_acc = -1.0
    for thr in thresholds:
        pred = (probs >= thr).astype(np.int64)
        acc = (pred == labels).mean()
        if acc > best_acc:
            best_acc = acc
            best_thr = float(thr)
    return best_thr, float(best_acc)

def apply_tta(batch: torch.Tensor, mode: str) -> torch.Tensor:
    if mode == "none":
        return batch
    if mode == "hflip":
        return torch.flip(batch, dims=[3])
    if mode == "vflip":
        return torch.flip(batch, dims=[2])
    if mode == "hvflip":
        return torch.flip(batch, dims=[2, 3])
    raise ValueError(f"Unknown TTA mode: {mode}")

@torch.no_grad()
def predict_loader(model: nn.Module, loader: DataLoader, use_tta: bool = False):
    model.eval()
    all_ids = []
    all_logits = []

    modes = TTA_MODES if use_tta else ("none",)

    for batch in tqdm(loader, leave=False):
        x = batch["image"].to(device, non_blocking=True)
        batch_logits = 0.0

        for mode in modes:
            x_tta = apply_tta(x, mode)
            with torch.autocast(device_type=device.type, enabled=amp_enabled, dtype=torch.float16):
                logits, _ = model(x_tta, grl_lambda=0.0)
            batch_logits = batch_logits + logits.float()

        batch_logits = batch_logits / len(modes)
        all_logits.append(batch_logits.cpu())
        all_ids.extend(batch["id"].tolist())

    all_logits = torch.cat(all_logits).numpy()
    return np.asarray(all_ids), all_logits

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader):
    ids, logits = predict_loader(model, loader, use_tta=False)
    labels = np.asarray(loader.dataset.labels, dtype=np.int64)
    thr, acc = sweep_best_threshold(logits, labels)
    loss = F.binary_cross_entropy_with_logits(
        torch.tensor(logits, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.float32)
    ).item()
    return {"ids": ids, "logits": logits, "labels": labels, "best_thr": thr, "acc": acc, "loss": loss}

In [17]:
train_dataset = H5PatchDataset(
    TRAIN_IMAGES_PATH,
    transform=TrainTransform(IMAGE_SIZE),
    return_label=True,
    return_center=True,
)

val_dataset = H5PatchDataset(
    VAL_IMAGES_PATH,
    transform=EvalTransform(IMAGE_SIZE),
    return_label=True,
    return_center=True,
)

test_dataset = H5PatchDataset(
    TEST_IMAGES_PATH,
    transform=EvalTransform(IMAGE_SIZE),
    return_label=False,
    return_center=False,
)

train_weights = compute_balanced_sample_weights(train_dataset.labels, train_dataset.centers)
train_sampler = WeightedRandomSampler(
    weights=train_weights,
    num_samples=len(train_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=NUM_WORKERS > 0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=NUM_WORKERS > 0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=NUM_WORKERS > 0,
)

center_values = sorted(np.unique(train_dataset.centers).tolist())
center_to_idx = {c: i for i, c in enumerate(center_values)}
print(f"Train centers: {center_values} -> mapped to {center_to_idx}")

Train centers: [0, 3, 4] -> mapped to {0: 0, 3: 1, 4: 2}


In [18]:
model = OODDinov3Classifier(MODEL_NAME, n_centers=len(center_values)).to(device)

optimizer = torch.optim.AdamW(
    [
        {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
        {"params": model.classifier.parameters(), "lr": LR_HEAD},
        {"params": model.domain_head.parameters(), "lr": LR_HEAD},
    ],
    weight_decay=WEIGHT_DECAY,
)

total_steps = math.ceil(len(train_loader) / ACCUMULATION_STEPS) * NUM_EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

## Train on train / validate on the held-out center

In [19]:
best_state = None
best_score = -1.0
best_epoch = -1
best_threshold = 0.5
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_cls = []
    running_dom = []
    running_acc = []

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), leave=False)
    for step, batch in pbar:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["label"].float().to(device, non_blocking=True)

        c = torch.tensor([center_to_idx[int(z)] for z in batch["center"]], device=device, dtype=torch.long)

        # Smooth schedule for the gradient reversal strength
        global_progress = (epoch * len(train_loader) + step + 1) / max(1, NUM_EPOCHS * len(train_loader))
        grl_lambda = float(2.0 / (1.0 + math.exp(-10.0 * global_progress)) - 1.0)

        with torch.autocast(device_type=device.type, enabled=amp_enabled, dtype=torch.float16):
            cls_logits, dom_logits = model(x, grl_lambda=grl_lambda)
            cls_loss = bce_with_label_smoothing(cls_logits, y, smoothing=LABEL_SMOOTHING)
            dom_loss = F.cross_entropy(dom_logits, c)
            loss = cls_loss + DOMAIN_ADV_WEIGHT * dom_loss
            loss = loss / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        probs = torch.sigmoid(cls_logits.detach())
        preds = (probs >= 0.5).float()
        acc = (preds == y).float().mean().item()

        running_cls.append(cls_loss.item())
        running_dom.append(dom_loss.item())
        running_acc.append(acc)

        pbar.set_description(
            f"epoch {epoch+1}/{NUM_EPOCHS} "
            f"cls={np.mean(running_cls):.4f} "
            f"dom={np.mean(running_dom):.4f} "
            f"acc@0.5={np.mean(running_acc):.4f}"
        )

    val_metrics = evaluate(model, val_loader)
    history.append({
        "epoch": epoch + 1,
        "train_cls_loss": float(np.mean(running_cls)),
        "train_dom_loss": float(np.mean(running_dom)),
        "train_acc@0.5": float(np.mean(running_acc)),
        "val_loss": float(val_metrics["loss"]),
        "val_best_acc": float(val_metrics["acc"]),
        "val_best_thr": float(val_metrics["best_thr"]),
    })

    print(
        f"Epoch {epoch+1:02d} | "
        f"train_cls={np.mean(running_cls):.4f} | "
        f"train_dom={np.mean(running_dom):.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc(best_thr)={val_metrics['acc']:.4f} | "
        f"best_thr={val_metrics['best_thr']:.3f}"
    )

    if val_metrics["acc"] > best_score:
        best_score = val_metrics["acc"]
        best_epoch = epoch
        best_threshold = val_metrics["best_thr"]
        best_state = {
            "model": copy.deepcopy(model.state_dict()),
            "epoch": epoch + 1,
            "val_acc": float(val_metrics["acc"]),
            "val_thr": float(val_metrics["best_thr"]),
            "config": {
                "model_name": MODEL_NAME,
                "image_size": IMAGE_SIZE,
                "batch_size": BATCH_SIZE,
                "domain_adv_weight": DOMAIN_ADV_WEIGHT,
            },
        }
        torch.save(best_state, SAVE_PATH)
        print(f"  -> saved new best model to {SAVE_PATH}")

    if epoch - best_epoch >= PATIENCE:
        print("Early stopping.")
        break

history_df = pd.DataFrame(history)
display(history_df)

  0%|          | 0/3125 [00:00<?, ?it/s]

  0%|          | 0/1091 [00:00<?, ?it/s]

Epoch 01 | train_cls=0.1251 | train_dom=0.6494 | val_loss=0.1352 | val_acc(best_thr)=0.9548 | best_thr=0.555
  -> saved new best model to best_dinov3_ood.pth


  0%|          | 0/3125 [00:00<?, ?it/s]

  0%|          | 0/1091 [00:00<?, ?it/s]

...

Epoch 04 | train_cls=0.0892 | train_dom=1.0731 | val_loss=0.1347 | val_acc(best_thr)=0.9545 | best_thr=0.475


  0%|          | 0/3125 [00:00<?, ?it/s]

  0%|          | 0/1091 [00:00<?, ?it/s]

Epoch 05 | train_cls=0.0855 | train_dom=1.0788 | val_loss=0.1513 | val_acc(best_thr)=0.9507 | best_thr=0.475
Early stopping.


,epoch,train_cls_loss,train_dom_loss,train_acc@0.5,val_loss,val_best_acc,val_best_thr
0,1,0.125105,0.649379,0.97278,0.135206,0.954848,0.555
1,2,0.100670,1.021656,0.98258,0.132488,0.958343,0.685
2,3,0.092486,1.059018,0.98617,0.151418,0.953043,0.715
3,4,0.089208,1.073089,0.98739,0.134748,0.954532,0.475
4,5,0.085543,1.078827,0.98858,0.151256,0.950665,0.475


## Load best checkpoint and generate validation diagnostics

In [20]:
ckpt = torch.load(SAVE_PATH, map_location=device)
model.load_state_dict(ckpt["model"])
best_threshold = float(ckpt["val_thr"])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} | val_acc={ckpt['val_acc']:.4f} | thr={best_threshold:.3f}")

val_ids, val_logits = predict_loader(model, val_loader, use_tta=False)
val_probs = 1.0 / (1.0 + np.exp(-val_logits))
val_preds = (val_probs >= best_threshold).astype(np.int64)
val_acc = (val_preds == val_dataset.labels).mean()

print(f"Validation accuracy with stored threshold: {val_acc:.4f}")

Loaded best checkpoint from epoch 2 | val_acc=0.9583 | thr=0.685


  0%|          | 0/1091 [00:00<?, ?it/s]

Validation accuracy with stored threshold: 0.9583


## Final test prediction

The competition expects integer predictions in two columns: `ID,Pred`.

In [21]:
test_ids, test_logits = predict_loader(model, test_loader, use_tta=USE_TTA)
test_probs = 1.0 / (1.0 + np.exp(-test_logits))
test_preds = (test_probs >= best_threshold).astype(np.int64)

submission = pd.DataFrame({
    "ID": test_ids.astype(int),
    "Pred": test_preds.astype(int),
}).sort_values("ID").reset_index(drop=True)

submission.to_csv(SUBMISSION_PATH, index=False)
print(submission.head(10))
print(f"Saved submission to {SUBMISSION_PATH}")

  0%|          | 0/2658 [00:00<?, ?it/s]

   ID  Pred
0   0     1
1   1     1
2   2     1
3   3     1
4   4     1
5   5     1
6   6     1
7   7     1
8   8     1
9   9     1
Saved submission to submission_dinov3_ood.csv
